In [ ]:
!pip -q install faster-whisper deep-translator sentencepiece "gradio>=5,<6" > /dev/null 2>&1

In [ ]:
import os, sys, glob, html, queue, threading, site, ctypes, time, logging, tempfile, subprocess, requests
import numpy as np, gradio as gr, ctranslate2
from faster_whisper import WhisperModel
from deep_translator import GoogleTranslator

logging.getLogger("huggingface_hub").setLevel(logging.ERROR)  # hide the HF token notice

# GPU check ----------
HAS_GPU = ctranslate2.get_cuda_device_count() > 0

def preload_cuda_libs():
    # Load CUDA 12 libs if they were installed with pip
    pats = ["nvidia/cublas/lib/libcublasLt.so.12*", "nvidia/cublas/lib/libcublas.so.12*",
            "nvidia/cudnn/lib/libcudnn.so.9*"]
    for pat in pats:
        for base in site.getsitepackages():
            for f in glob.glob(os.path.join(base, pat)):
                try:
                    ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
                except OSError:
                    pass

def gpu_works():
    # Quick test on 3 seconds of silence
    try:
        m = WhisperModel("tiny", device="cuda", compute_type="float16")
        segs, _ = m.transcribe(np.zeros(16000 * 3, dtype=np.float32))
        list(segs)
        return True
    except Exception:
        return False

if HAS_GPU:
    preload_cuda_libs()
    if not gpu_works():
        print("Fixing the GPU setup. This can take a few minutes...")
        subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                        "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"],
                       check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        preload_cuda_libs()
        HAS_GPU = gpu_works()
print("GPU is ready." if HAS_GPU else "No GPU. Using the CPU.")

#  Languages Whisper can hear ----------
WHISPER_LANGS = {
 "en":"English","zh":"Chinese","de":"German","es":"Spanish","ru":"Russian","ko":"Korean","fr":"French",
 "ja":"Japanese","pt":"Portuguese","tr":"Turkish","pl":"Polish","ca":"Catalan","nl":"Dutch","ar":"Arabic",
 "sv":"Swedish","it":"Italian","id":"Indonesian","hi":"Hindi","fi":"Finnish","vi":"Vietnamese","he":"Hebrew",
 "uk":"Ukrainian","el":"Greek","ms":"Malay","cs":"Czech","ro":"Romanian","da":"Danish","hu":"Hungarian",
 "ta":"Tamil","no":"Norwegian","th":"Thai","ur":"Urdu","hr":"Croatian","bg":"Bulgarian","lt":"Lithuanian",
 "la":"Latin","mi":"Maori","ml":"Malayalam","cy":"Welsh","sk":"Slovak","te":"Telugu","fa":"Persian",
 "lv":"Latvian","bn":"Bengali","sr":"Serbian","az":"Azerbaijani","sl":"Slovenian","kn":"Kannada",
 "et":"Estonian","mk":"Macedonian","br":"Breton","eu":"Basque","is":"Icelandic","hy":"Armenian",
 "ne":"Nepali","mn":"Mongolian","bs":"Bosnian","kk":"Kazakh","sq":"Albanian","sw":"Swahili","gl":"Galician",
 "mr":"Marathi","pa":"Punjabi","si":"Sinhala","km":"Khmer","sn":"Shona","yo":"Yoruba","so":"Somali",
 "af":"Afrikaans","oc":"Occitan","ka":"Georgian","be":"Belarusian","tg":"Tajik","sd":"Sindhi","gu":"Gujarati",
 "am":"Amharic","yi":"Yiddish","lo":"Lao","uz":"Uzbek","fo":"Faroese","ht":"Haitian Creole","ps":"Pashto",
 "tk":"Turkmen","nn":"Nynorsk","mt":"Maltese","sa":"Sanskrit","lb":"Luxembourgish","my":"Myanmar",
 "bo":"Tibetan","tl":"Tagalog","mg":"Malagasy","as":"Assamese","tt":"Tatar","haw":"Hawaiian","ln":"Lingala",
 "ha":"Hausa","ba":"Bashkir","jw":"Javanese","su":"Sundanese","yue":"Cantonese",
}
NAME2CODE = {n: c for c, n in WHISPER_LANGS.items()}
AUTO = "Auto-detect (works with mixed languages)"
SPOKEN_CHOICES = [AUTO] + sorted(NAME2CODE)

# ---------- Whisper ----------
_models = {}

def get_model(name, device):
    key = (name, device)
    if key not in _models:
        if device == "cuda":
            _models[key] = WhisperModel(name, device="cuda", compute_type="float16")
        else:
            _models[key] = WhisperModel(name, device="cpu", compute_type="int8",
                                        cpu_threads=os.cpu_count() or 2)
    return _models[key]

def load_audio(path):
    # ffmpeg pulls the sound out of almost any audio or video file
    try:
        cmd = ["ffmpeg", "-nostdin", "-threads", "0", "-i", path, "-vn",
               "-f", "s16le", "-ac", "1", "-acodec", "pcm_s16le", "-ar", "16000", "-"]
        out = subprocess.run(cmd, capture_output=True, check=True).stdout
        return np.frombuffer(out, np.int16).astype(np.float32) / 32768.0
    except Exception:
        return path

# - Translation ----------
LANGS = GoogleTranslator().get_supported_languages(as_dict=True)
LANG_CHOICES = ["(No translation)"] + [l.title() for l in LANGS]

NLLB_CODES = {
 "afrikaans":"afr_Latn","amharic":"amh_Ethi","arabic":"arb_Arab","bengali":"ben_Beng","bulgarian":"bul_Cyrl",
 "catalan":"cat_Latn","chinese (simplified)":"zho_Hans","chinese (traditional)":"zho_Hant","croatian":"hrv_Latn",
 "czech":"ces_Latn","danish":"dan_Latn","dutch":"nld_Latn","estonian":"est_Latn","filipino":"tgl_Latn",
 "finnish":"fin_Latn","french":"fra_Latn","german":"deu_Latn","greek":"ell_Grek","gujarati":"guj_Gujr",
 "hausa":"hau_Latn","hebrew":"heb_Hebr","hindi":"hin_Deva","hungarian":"hun_Latn","icelandic":"isl_Latn",
 "igbo":"ibo_Latn","indonesian":"ind_Latn","italian":"ita_Latn","japanese":"jpn_Jpan","kannada":"kan_Knda",
 "korean":"kor_Hang","latvian":"lvs_Latn","lithuanian":"lit_Latn","malay":"zsm_Latn","malayalam":"mal_Mlym",
 "marathi":"mar_Deva","nepali":"npi_Deva","norwegian":"nob_Latn","oromo":"gaz_Latn","persian":"pes_Arab",
 "polish":"pol_Latn","portuguese":"por_Latn","punjabi":"pan_Guru","romanian":"ron_Latn","russian":"rus_Cyrl",
 "serbian":"srp_Cyrl","sinhala":"sin_Sinh","slovak":"slk_Latn","slovenian":"slv_Latn","somali":"som_Latn",
 "spanish":"spa_Latn","swahili":"swh_Latn","swedish":"swe_Latn","tamil":"tam_Taml","telugu":"tel_Telu",
 "thai":"tha_Thai","tigrinya":"tir_Ethi","turkish":"tur_Latn","ukrainian":"ukr_Cyrl","urdu":"urd_Arab",
 "vietnamese":"vie_Latn","yoruba":"yor_Latn","zulu":"zul_Latn",
}

NLLB_CODES.update({
 "english":"eng_Latn","chinese":"zho_Hans","tagalog":"tgl_Latn","myanmar":"mya_Mymr","georgian":"kat_Geor",
 "kazakh":"kaz_Cyrl","uzbek":"uzn_Latn","khmer":"khm_Khmr","lao":"lao_Laoo","mongolian":"khk_Cyrl",
 "pashto":"pbt_Arab","sindhi":"snd_Arab","sanskrit":"san_Deva","armenian":"hye_Armn","azerbaijani":"azj_Latn",
 "basque":"eus_Latn","belarusian":"bel_Cyrl","bosnian":"bos_Latn","galician":"glg_Latn","javanese":"jav_Latn",
 "sundanese":"sun_Latn","macedonian":"mkd_Cyrl","maltese":"mlt_Latn","welsh":"cym_Latn","shona":"sna_Latn",
 "lingala":"lin_Latn","malagasy":"plt_Latn","albanian":"als_Latn","assamese":"asm_Beng","tajik":"tgk_Cyrl",
 "tatar":"tat_Cyrl","turkmen":"tuk_Latn","luxembourgish":"ltz_Latn","occitan":"oci_Latn","tibetan":"bod_Tibt",
 "faroese":"fao_Latn","haitian creole":"hat_Latn","bashkir":"bak_Cyrl","nynorsk":"nno_Latn",
})

def _gtx(text, code, delays=(2, 4)):
    # Google's web translate. It guesses the source language by itself
    for d in (*delays, None):
        r = requests.post("https://translate.googleapis.com/translate_a/single",
                          params={"client": "gtx", "sl": "auto", "tl": code, "dt": "t"},
                          data={"q": text}, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        if r.status_code == 200:
            return "".join(seg[0] for seg in r.json()[0] if seg and seg[0])
        if r.status_code == 429 and d is not None:
            time.sleep(d)
            continue
        raise RuntimeError(f"Google error {r.status_code}")

def google_translate(lines, code, progress=None):
    # Send big chunks, not one line at a time
    out, chunk, size = [], [], 0
    def flush():
        nonlocal chunk, size
        if not chunk:
            return
        parts = _gtx("\n".join(chunk), code).split("\n")
        if len(parts) != len(chunk):          # lines got mixed up, so do them one by one
            parts = []
            for l in chunk:
                parts.append(_gtx(l, code))
                time.sleep(0.3)
        out.extend(parts)
        chunk, size = [], 0
        if progress:
            progress(0.9 + 0.1 * len(out) / len(lines), desc=f"Translating... {len(out)}/{len(lines)}")
    for l in lines:
        if size + len(l) > 1500:
            flush()
        chunk.append(l.replace("\n", " "))
        size += len(l) + 1
    flush()
    return out

_nllb = {}

def nllb_translate(lines, src_code, tgt_code, model_choice, progress):
    # Offline translator. Freeee but the first download is big
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    name = "facebook/nllb-200-distilled-1.3B" if "1.3B" in model_choice else "facebook/nllb-200-distilled-600M"
    dev = "cuda" if (HAS_GPU and torch.cuda.is_available()) else "cpu"
    if name not in _nllb:
        progress(0.9, desc="Getting the offline translator ready (first time takes a while)...")
        tok = AutoTokenizer.from_pretrained(name)
        model = AutoModelForSeq2SeqLM.from_pretrained(
            name, low_cpu_mem_usage=True,
            torch_dtype=torch.float16 if dev == "cuda" else torch.float32).eval()
        try:
            model = model.to(dev)
        except Exception:
            dev, model = "cpu", model.float().to("cpu")
        _nllb[name] = (tok, model, dev)
    tok, model, dev = _nllb[name]
    tok.src_lang = src_code
    bs = 16 if dev == "cuda" else 6
    out = []
    for k in range(0, len(lines), bs):
        batch = lines[k:k + bs]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=200).to(dev)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(tgt_code),
                                 max_new_tokens=256, num_beams=4)
        out.extend(tok.batch_decode(gen, skip_special_tokens=True))
        progress(0.9 + 0.1 * min(1.0, (k + bs) / len(lines)),
                 desc=f"Translating... {min(k + bs, len(lines))}/{len(lines)}")
    return out

def translate(lines, src_name, lang_name, nllb_choice, progress):
    key = lang_name.lower()
    try:
        return google_translate(lines, LANGS[key], progress)
    except Exception as e:
        print("Google failed:", e)
        progress(0.9, desc="Google is busy. Switching to the offline translator...")
    src = NLLB_CODES.get(src_name.lower())
    tgt = NLLB_CODES.get(key)
    if not src or not tgt:
        raise RuntimeError("The offline translator doesn't know one of these languages.")
    return nllb_translate(lines, src, tgt, nllb_choice, progress)

# ---------- Files ----------
def ts(sec):
    ms = int(sec * 1000)
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

def make_srt(segs, lines):
    return "\n".join(f"{i}\n{ts(a)} --> {ts(b)}\n{line}\n"
                     for i, ((a, b, _), line) in enumerate(zip(segs, lines), 1))

def save(folder, name, content):
    path = os.path.join(folder, name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return path

# ---------- Progress bar ----------
def bar(pct, label, done=False, wait=False, error=False):
    # pct=None: no number. wait=True: moving bar for slow steps. error=True: red bar
    if error:
        num, width = "", 100
    else:
        num = "" if pct is None else f"<b>{int(max(0, min(100, pct)))}%</b>"
        width = 0 if pct is None else int(max(0, min(100, pct)))
    cls = "pfill" + (" done" if done else "") + (" wait" if wait else "") + (" err" if error else "")
    return (f"<div class='pwrap'><div class='ptop'><span>{html.escape(label)}</span>{num}</div>"
            f"<div class='pbar'><div class='{cls}' style='width:{width}%'></div></div></div>")

def clock(sec):
    m, s = divmod(int(sec), 60)
    return f"{m}:{s:02d}"

class Tracker:

    def __init__(self):
        self.frac, self.desc = 0.0, ""
    def __call__(self, frac=None, desc=None):
        if frac is not None:
            self.frac = frac
        if desc:
            self.desc = desc


HIDE = gr.update(visible=False)

def pack(text=None, tr=None, btns=None, barhtml=None):
    # None means "leave this box as it is"
    keep = gr.update()
    return [keep if text is None else text, keep if tr is None else tr,
            *(btns if btns else [keep] * 4), keep if barhtml is None else barhtml]

def with_heartbeat(fn, label):

    box, t0 = {}, time.time()
    def work():
        try:
            box["v"] = fn()
        except Exception as e:
            box["e"] = e
    th = threading.Thread(target=work, daemon=True)
    th.start()
    while th.is_alive():
        yield pack(barhtml=bar(None, f"{label} ({clock(time.time() - t0)})", wait=True))
        time.sleep(0.5)
    th.join()
    if "e" in box:
        raise box["e"]
    return box["v"]

def _run(file, mic, spoken, model_name, target_lang, nllb_choice, skip_silence):
    src = file or mic
    if not src:
        raise gr.Error("Please add an audio or video file first.")
    print("Started:", os.path.basename(src))
    want_tr = target_lang != "(No translation)"
    top = 70 if want_tr else 98          # where the "listening" part of the bar ends
    yield pack("", "", [HIDE] * 4, bar(2, "Reading your file..."))

    device = "cuda" if HAS_GPU else "cpu"   # picked automatically
    if device == "cpu":
        gr.Warning("No GPU found, so this will be slow. A short clip is a good test.")
    lang_code = None if spoken == AUTO else NAME2CODE[spoken]
    audio_data = yield from with_heartbeat(lambda: load_audio(src), "Reading your file")

    def listen(dev, use_vad):
        model = yield from with_heartbeat(lambda: get_model(model_name, dev),
                                          f"Loading the model on the {dev.upper()}. The first time is slow")

        vad_opts = dict(threshold=0.3, min_silence_duration_ms=2000, speech_pad_ms=500) if use_vad else None
        q = queue.Queue()

        def produce():

            try:
                opts = dict(language=lang_code, beam_size=1 if dev == "cpu" else 5,
                            vad_filter=use_vad, vad_parameters=vad_opts,
                            condition_on_previous_text=False,   # stops repeated lines
                            no_speech_threshold=None)           # don't skip unsure parts
                if lang_code is None:
                    opts["multilingual"] = True                 # check the language again and again
                try:
                    segments, info = model.transcribe(audio_data, **opts)
                except TypeError:                               # older version without this option
                    opts.pop("multilingual", None)
                    segments, info = model.transcribe(audio_data, **opts)
                q.put(("info", info))
                for s in segments:
                    q.put(("seg", s))
                q.put(("end", None))
            except Exception as e:
                q.put(("err", e))

        threading.Thread(target=produce, daemon=True).start()
        t0, last_yield, info, out = time.time(), 0.0, None, []
        pct, label = 10, "Getting the audio ready..."
        while True:
            try:
                kind, val = q.get(timeout=0.5)
            except queue.Empty:
                if info:
                    yield pack(barhtml=bar(pct, f"{label} ({clock(time.time() - t0)})"))
                else:
                    yield pack(barhtml=bar(None, f"{label} ({clock(time.time() - t0)})", wait=True))
                continue
            if kind == "info":
                info = val
            elif kind == "seg":
                out.append((val.start, val.end, val.text.strip()))
                if info.duration:
                    pct = 10 + (top - 10) * min(val.end / info.duration, 1.0)
                    label = f"Listening... {int(val.end)}s of {int(info.duration)}s"
                if time.time() - last_yield > 0.3:
                    last_yield = time.time()
                    live = " ".join(t for _, _, t in out if t)       # show the text as it comes
                    yield pack(live, None, None, bar(pct, label))
            elif kind == "err":
                raise val
            else:
                break
        return out, info.language, info.duration

    def listen_all(dev):
        out, lang, total = yield from listen(dev, skip_silence)
        # If much of the audio is missing, try again without skipping silence
        if skip_silence and total > 30 and (not out or out[-1][1] < total * 0.6):
            yield pack("", None, None, bar(8, "Some parts may be missing. Trying again..."))
            out, lang, total = yield from listen(dev, False)
        return out, lang

    try:
        segs, found = yield from listen_all(device)
    except Exception:
        if device != "cuda":
            raise
        gr.Warning("The GPU had a problem. Trying again on the CPU...")
        segs, found = yield from listen_all("cpu")

    src_name = WHISPER_LANGS.get(found, found)
    if lang_code is None:
        gr.Info(f"The audio starts in {src_name}.")

    texts = [t for _, _, t in segs if t]
    segs = [s for s in segs if s[2]]
    if not texts:
        yield pack("I couldn't hear any speech in this file.", "", [HIDE] * 4, bar(100, "No speech found", True))
        return

    text = " ".join(texts)
    base = os.path.splitext(os.path.basename(src))[0] or "audio"
    folder = tempfile.mkdtemp()
    txt_file = save(folder, f"{base}.txt", text)
    srt_file = save(folder, f"{base}.srt", make_srt(segs, texts))
    btns = [gr.update(value=txt_file, visible=True), gr.update(value=srt_file, visible=True), HIDE, HIDE]

    if not want_tr:
        yield pack(text, "", btns, bar(100, "Done ✅", True))
        return
    if lang_code is not None and target_lang.lower() == src_name.lower():
        gr.Info("The audio is already in that language.")
        yield pack(text, "", btns, bar(100, "Done ✅", True))
        return

    # Translate in the background so the bar keeps moving
    yield pack(text, "⏳ Getting ready to translate...", btns, bar(top, f"Translating to {target_lang}..."))
    tracker, box = Tracker(), {}
    def work():
        try:
            box["tr"] = translate(texts, src_name, target_lang, nllb_choice, tracker)
        except Exception as e:
            box["err"] = e
    th = threading.Thread(target=work, daemon=True)
    th.start()
    t0 = time.time()
    while th.is_alive():
        t = min(max((tracker.frac - 0.9) / 0.1, 0.0), 1.0)
        label = tracker.desc or f"Translating to {target_lang}..."
        yield pack(None, "⏳ " + label, None, bar(top + (98 - top) * t, f"{label} ({clock(time.time() - t0)})"))
        time.sleep(0.4)
    th.join()

    if "err" in box:
        print("Translation failed:", box["err"])
        gr.Warning("Couldn't translate this time. Your text is ready. Try again in a minute.")
        yield pack(None, "Couldn't translate this time. Try again in a minute.", None,
                   bar(100, "Translation failed. Your text is ready.", error=True))
        return
    translated = " ".join(box["tr"])
    tag = target_lang.lower().replace(" ", "_")
    btns[2] = gr.update(value=save(folder, f"{base}_{tag}.txt", translated), visible=True)
    btns[3] = gr.update(value=save(folder, f"{base}_{tag}.srt", make_srt(segs, box["tr"])), visible=True)
    yield pack(text, translated, btns, bar(100, "Done ✅", True))

def run(*args):
    # If anything breaks, say so in the bar instead of staying silent
    try:
        yield from _run(*args)
    except gr.Error:
        raise
    except Exception as e:
        import traceback
        traceback.print_exc()
        gr.Warning("Something went wrong. See the red bar.")
        yield pack(None, None, None, bar(100, f"Something went wrong: {str(e)[:160]}", error=True))

def on_upload(f):
    # Just a note that the file arrived. The progress bar is only for the work itself
    if not f:
        return ""
    mb = os.path.getsize(f) / 1e6
    return f"<div class='pwrap'>✅ File ready: {html.escape(os.path.basename(f))} ({mb:.1f} MB). Press Transcribe.</div>"

# ---------- Upload progress (runs in your browser) ----------

UPLOAD_JS = r'''
() => {
  if (window.__uploadBarPatched) return;
  window.__uploadBarPatched = true;
  var hideTimer = null;
  var UPLOAD = /\/upload(\?|$)/;
  function mb(b) { return (b / 1048576).toFixed(1); }
  function box() { return document.getElementById('up-progress'); }
  function hide() { var el = box(); if (el) el.style.display = 'none'; }
  function paint(loaded, total, done) {
    var el = box();
    if (!el) return;
    clearTimeout(hideTimer);
    var known = total > 0;
    var pct = done ? 100 : (known ? Math.min(99, Math.round(loaded * 100 / total)) : 0);
    var label = done ? 'Uploaded ✅' :
      'Uploading your file... ' + (known ? mb(loaded) + ' of ' + mb(total) + ' MB' : mb(loaded) + ' MB');
    el.style.display = 'block';
    el.innerHTML = '<div class="ptop"><span>' + label + '</span><b>' + pct + '%</b></div>' +
      '<div class="pbar"><div class="pfill' + (done ? ' done' : '') + '" style="width:' + pct + '%"></div></div>';
    if (done) hideTimer = setTimeout(hide, 2500);
  }

  // Case 1: the page uses XMLHttpRequest for uploads
  var origOpen = XMLHttpRequest.prototype.open;
  var origSend = XMLHttpRequest.prototype.send;
  XMLHttpRequest.prototype.open = function (method, url) {
    this.__url = String(url);
    return origOpen.apply(this, arguments);
  };
  XMLHttpRequest.prototype.send = function () {
    var xhr = this;
    try {
      if (UPLOAD.test(xhr.__url || '') && xhr.upload && !xhr.__mine) {
        paint(0, 0, false);
        xhr.upload.addEventListener('progress', function (e) { paint(e.loaded, e.lengthComputable ? e.total : 0, false); });
        xhr.addEventListener('load', function () { paint(0, 0, true); });
        xhr.addEventListener('error', hide);
        xhr.addEventListener('abort', hide);
      }
    } catch (err) {}
    return origSend.apply(xhr, arguments);
  };

  // Case 2: the page uses fetch. fetch can't report upload progress,
  // so for uploads we send the same request with XMLHttpRequest instead.
  var origFetch = window.fetch ? window.fetch.bind(window) : null;
  if (origFetch) {
    window.fetch = function (input, init) {
      try {
        var url = typeof input === 'string' ? input : ((input && input.url) || String(input));
        var method = ((init && init.method) || (input && input.method) || 'GET').toUpperCase();
        var body = init && init.body;
        if (method === 'POST' && UPLOAD.test(url) && typeof FormData !== 'undefined' && body instanceof FormData) {
          return new Promise(function (resolve, reject) {
            var xhr = new XMLHttpRequest();
            xhr.__mine = true;
            origOpen.call(xhr, 'POST', url);
            try {
              if (init.headers) {
                new Headers(init.headers).forEach(function (v, k) { xhr.setRequestHeader(k, v); });
              }
            } catch (err) {}
            xhr.upload.onprogress = function (e) { paint(e.loaded, e.lengthComputable ? e.total : 0, false); };
            xhr.onload = function () {
              paint(0, 0, true);
              try {
                resolve(new Response(xhr.responseText, {
                  status: xhr.status, statusText: xhr.statusText,
                  headers: { 'Content-Type': xhr.getResponseHeader('Content-Type') || 'application/json' }
                }));
              } catch (err) {
                resolve(origFetch(input, init));   // last resort: do the upload the normal way
              }
            };
            xhr.onerror = function () { hide(); reject(new TypeError('Network error')); };
            xhr.onabort = function () { hide(); reject(new DOMException('Aborted', 'AbortError')); };
            if (init.signal) init.signal.addEventListener('abort', function () { xhr.abort(); });
            paint(0, 0, false);
            origSend.call(xhr, body);
          });
        }
      } catch (err) {}
      return origFetch(input, init);
    };
  }
}
'''

# ---------- Look ----------
css = """
.gradio-container {max-width: 950px !important; margin: auto;}
#title {text-align:center; background: linear-gradient(90deg,#6366f1,#ec4899);
        -webkit-background-clip: text; -webkit-text-fill-color: transparent; font-size: 2.2em; font-weight: 800;}
#go {background: linear-gradient(90deg,#6366f1,#ec4899); color: white; border: none; font-size: 1.1em;}
.pwrap {padding: 10px 12px; border-radius: 12px; border: 1px solid rgba(128,128,128,.3);}
.ptop {display:flex; justify-content:space-between; font-size:.9em; margin-bottom:6px;}
.pbar {height:12px; border-radius:99px; background: rgba(128,128,128,.25); overflow:hidden;}
.pfill {height:100%; border-radius:99px; transition: width .35s ease;
        background: linear-gradient(90deg,#6366f1,#ec4899,#6366f1); background-size: 200% 100%;
        animation: slide 1.4s linear infinite;}
.pfill.done {animation:none; background: linear-gradient(90deg,#22c55e,#16a34a);}
.pfill.err {animation:none; background: linear-gradient(90deg,#ef4444,#b91c1c);}
.pfill.wait {width:35% !important; animation: sweep 1.1s ease-in-out infinite alternate;}
@keyframes sweep {from {margin-left: 0;} to {margin-left: 65%;}}
@keyframes slide {from {background-position: 0% 0;} to {background-position: -200% 0;}}
"""

status = "GPU is ready" if HAS_GPU else "No GPU found, so it runs on the CPU. It will be slower."

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="pink"), css=css, js=UPLOAD_JS, title="Audio & Video to Text") as demo:
    gr.HTML("<div id='title'>🎙️ Audio & Video to Text</div>")
    gr.Markdown(f"<center>Add audio or video in any of 99 supported languages by faster-whisper. Get  text and translate it if you want...<br><b>{status}</b></center>")
    with gr.Row():
        with gr.Column():
            with gr.Tabs():
                with gr.Tab("📁 Upload a file"):
                    file_in = gr.File(label="Audio or video file", file_types=["audio", "video"], type="filepath")
                with gr.Tab("🎤 Record"):
                    mic_in = gr.Audio(sources=["microphone"], type="filepath", label="Record with your mic")
            gr.HTML("<div id='up-progress' class='pwrap' style='display:none'></div>")   # filled by the script above
            upload_html = gr.HTML()
            spoken = gr.Dropdown(SPOKEN_CHOICES, value=AUTO, label="🗣️ Language in the audio")
            target = gr.Dropdown(LANG_CHOICES, value="(No translation)", label="🌍 Translate to")
            model_name = gr.Dropdown([("Turbo - fast and very accurate", "large-v3-turbo"),
                                      ("Large - the most accurate, but slower", "large-v3")],
                                     value="large-v3-turbo", label="🧠 Model")
            with gr.Accordion("More options", open=False):
                skip_silence = gr.Checkbox(value=True, label="Skip silent parts (turn off if some text is missing)")
                nllb_choice = gr.Radio(["Fast (600M)", "Better (1.3B)"], value="Better (1.3B)",
                                       label="Backup translator (used when Google is busy)")
            btn = gr.Button("✨ Transcribe", elem_id="go", variant="primary")
        with gr.Column():
            out_text = gr.Textbox(label="📝 Text", lines=8, show_copy_button=True)
            progress_html = gr.HTML(value=bar(0, "Waiting for your file..."))
            out_tr = gr.Textbox(label="🌐 Translation", lines=8, show_copy_button=True)
            with gr.Row():
                b_txt = gr.DownloadButton("⬇️ Text (.txt)", visible=False)
                b_srt = gr.DownloadButton("⬇️ Subtitles (.srt)", visible=False)
            with gr.Row():
                b_tr_txt = gr.DownloadButton("⬇️ Translation (.txt)", visible=False)
                b_tr_srt = gr.DownloadButton("⬇️ Translated subtitles (.srt)", visible=False)
    btn.click(run, [file_in, mic_in, spoken, model_name, target, nllb_choice, skip_silence],
              [out_text, out_tr, b_txt, b_srt, b_tr_txt, b_tr_srt, progress_html],
              show_progress="hidden")
    file_in.upload(on_upload, file_in, upload_html, show_progress="hidden")
    file_in.clear(lambda: "", None, upload_html, show_progress="hidden")

demo.queue().launch(share=True, debug=True)